In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

In [2]:
DATA_PATH = 'Data/train.csv'

def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df = df.dropna(subset=['target']).reset_index(drop=True)
    return df

df = load_data()

In [7]:
def train_val_split(df, val_size=0.2):

    dates = np.sort(df['date_id'].unique())
    n_val = int(len(dates) * val_size)
    cutoff_date = dates[-n_val]

    train_df = df[df['date_id'] < cutoff_date].reset_index(drop=True)
    val_df = df[df['date_id'] >= cutoff_date].reset_index(drop=True)

    print(f"Train set: {train_df.date_id.nunique()} samples, Validation set: {val_df.date_id.nunique()} samples")

    return train_df, val_df

train_df, val_df = train_val_split(df, val_size=0.2)

Train set: 385 samples, Validation set: 96 samples


In [8]:
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

In [9]:
baseline_zero = mae(val_df['target'], np.zeros(len(val_df)))
print(f"Zero-prediction MAE on validation: {baseline_zero:.4f}")

Zero-prediction MAE on validation: 6.0601


## Linear Model

In [10]:
FEATURES = [
    'imbalance_buy_sell_flag', 'signed_imb', 'imb_ratio', 'book_imb',
    'near_minus_wap', 'ref_minus_wap', 'spread',
    'imbalance_size', 'matched_size', 'bid_size', 'ask_size',
    'wap', 'reference_price', 'seconds_in_bucket', 'stock_id',
]

def build_features(df):
    df = df.copy()
    df['signed_imb']     = df['imbalance_size'] * df['imbalance_buy_sell_flag']
    df['imb_ratio']      = df['signed_imb'] / df['matched_size']
    df['book_imb']       = (df['bid_size'] - df['ask_size']) / (df['bid_size'] + df['ask_size'])
    df['near_minus_wap'] = df['near_price'] - df['wap']       # NaN before 300s — left as-is for LGBM
    df['ref_minus_wap']  = df['reference_price'] - df['wap']
    df['spread']         = df['ask_price'] - df['bid_price']
    return df

train_fe = build_features(train_df)
val_fe   = build_features(val_df)

y_train, y_val = train_fe['target'], val_fe['target']

In [11]:
train_fe.columns

Index(['stock_id', 'date_id', 'seconds_in_bucket', 'imbalance_size',
       'imbalance_buy_sell_flag', 'reference_price', 'matched_size',
       'far_price', 'near_price', 'bid_price', 'bid_size', 'ask_price',
       'ask_size', 'wap', 'target', 'time_id', 'row_id', 'signed_imb',
       'imb_ratio', 'book_imb', 'near_minus_wap', 'ref_minus_wap', 'spread'],
      dtype='object')

In [12]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [14]:
linear_features = ['stock_id', 'seconds_in_bucket', 'imbalance_size','imbalance_buy_sell_flag', 'signed_imb', 'imb_ratio', 'book_imb', 'near_minus_wap', 'ref_minus_wap', 'spread', 'matched_size', 'bid_size', 'ask_size', 'wap', 'reference_price']

def clean_for_linear(df, features):
    df = df.copy()

    X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

    return X

X_train_lin = clean_for_linear(train_fe, linear_features)
X_val_lin   = clean_for_linear(val_fe,   linear_features)



In [15]:
scaler = StandardScaler()
X_train_lin_scaled = scaler.fit_transform(X_train_lin)
X_val_lin_scaled   = scaler.transform(X_val_lin)

In [16]:
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_lin_scaled, y_train)
y_pred_ridge = ridge.predict(X_val_lin_scaled)
pred_mae = mae(y_val, y_pred_ridge)
print(f"Ridge Regression MAE on validation: {pred_mae:.4f}")

Ridge Regression MAE on validation: 6.0306


In [17]:
# get weights of the ridge regression model
ridge_weights = pd.Series(ridge.coef_, index=X_train_lin.columns).sort_values(key=abs, ascending=False)
ridge_weights

ref_minus_wap              0.941320
book_imb                  -0.825049
imbalance_buy_sell_flag   -0.221022
near_minus_wap             0.180184
signed_imb                 0.178185
ask_size                  -0.133933
bid_size                   0.129111
wap                       -0.116686
reference_price           -0.046010
spread                     0.042631
seconds_in_bucket         -0.031629
imb_ratio                  0.024192
imbalance_size             0.020684
matched_size               0.009536
stock_id                   0.008006
dtype: float64

## LightGBM

In [19]:
import lightgbm as lgb

In [20]:
X_train = train_fe[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = val_fe[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)

In [22]:
model = lgb.LGBMRegressor(
    objective='mae',          # optimize what we're scored on
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=128,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    min_child_samples=100,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='mae',
    categorical_feature=['stock_id'],
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True)]
)

pred_lgb = model.predict(X_val)
lgb_mae = mae(y_val, pred_lgb)
print(f"LightGBM MAE on validation: {lgb_mae:.4f}")

/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015169 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3318
[LightGBM] [Info] Number of data points in the train set: 4181948, number of used features: 15
[LightGBM] [Info] Start training from score -0.069737
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[278]	valid_0's l1: 5.95528
LightGBM MAE on validation: 5.9553


## More Feature engineering

In [23]:
def add_price_features(df):
    df = df.copy()
    # A synthetic "mid" of the order book — often a better anchor than wap alone
    df['mid_price']   = (df['bid_price'] + df['ask_price']) / 2

    # Gaps between the auction's price views and the book (your validated signals)
    df['ref_wap']     = df['reference_price'] - df['wap']
    df['near_wap']    = df['near_price']      - df['wap']       # NaN < 300s
    df['far_near']    = df['far_price']       - df['near_price']# NaN < 300s
    df['far_wap']     = df['far_price']       - df['wap']

    # Where does reference sit inside the bid-ask? (0=at bid, 1=at ask)
    df['spread']      = df['ask_price'] - df['bid_price']
    df['ref_depth']   = (df['reference_price'] - df['bid_price']) / df['spread']
    df['wap_depth']   = (df['wap'] - df['bid_price']) / df['spread']
    return df

In [24]:
def add_size_features(df):
    df = df.copy()
    df['imb_to_matched'] = df['imbalance_size'] / df['matched_size']   # ⭐ your best ratio
    df['book_total']     = df['bid_size'] + df['ask_size']
    df['book_imb']       = (df['bid_size'] - df['ask_size']) / df['book_total']  # ⭐ strongest raw signal
    df['auction_vs_book']= df['matched_size'] / df['book_total']       # auction weight vs continuous
    df['imb_flag_size']  = df['imbalance_size'] * df['imbalance_buy_sell_flag']  # signed imbalance
    return df

In [25]:
def add_time_features(df):
    df = df.sort_values(['stock_id', 'date_id', 'seconds_in_bucket']).copy()
    g = df.groupby(['stock_id', 'date_id'], observed=True)

    for col in ['wap', 'imbalance_size', 'reference_price', 'imb_flag_size', 'matched_size']:
        df[f'{col}_diff']   = g[col].diff()          # change since previous (10s) snapshot
        df[f'{col}_ret']    = g[col].pct_change()    # % change, scale-free

    # Momentum: change over a longer window (e.g. last 60s = 6 steps)
    df['wap_mom_60']   = df['wap'] - g['wap'].shift(6)
    df['imb_mom_60']   = df['imbalance_size'] - g['imbalance_size'].shift(6)

    # How far into the auction are we? (context for everything else)
    df['auction_frac'] = df['seconds_in_bucket'] / 540
    return df

In [26]:
def add_cross_sectional(df):
    df = df.copy()
    grp = df.groupby(['date_id', 'seconds_in_bucket'], observed=True)

    for col in ['imb_flag_size', 'book_imb', 'near_wap', 'imb_to_matched']:
        # rank in [0,1] among peers at this instant
        df[f'{col}_rank'] = grp[col].rank(pct=True)
        # z-score vs peers: (x - mean) / std across stocks now
        df[f'{col}_z']    = (df[col] - grp[col].transform('mean')) / grp[col].transform('std')
    return df

In [43]:

def add_historical_features(df, windows=(5, 20)):
    df = df.copy()

    # ── Step 1: one summary row per (stock, date) from that day's data ────
    daily = df.groupby(['stock_id', 'date_id']).agg(
        day_target_std     = ('target', 'std'),          # how twitchy was this stock today
        day_target_absmean = ('target', lambda s: s.abs().mean()),  # avg move magnitude
        day_wap_vol        = ('wap', 'std'),             # realized price volatility
        day_imb_mean       = ('imbalance_size', 'mean'), # typical auction imbalance
    ).reset_index()

    # ── Step 2: roll each summary over PAST days, per stock ──────────────

    daily = daily.sort_values(['stock_id', 'date_id'])
    g = daily.groupby('stock_id')
    base_cols = ['day_target_std', 'day_target_absmean', 'day_wap_vol', 'day_imb_mean']

    for col in base_cols:
        daily[f'{col}_lag1'] = g[col].shift(1)           # simplest: yesterday's value
        for w in windows:                                # rolling mean of last w days
            daily[f'{col}_r{w}'] = g[col].transform(
                lambda s: s.rolling(w, min_periods=1).mean().shift(1)
            )

    # ── Step 3: merge the day-level history back onto every intraday row ─
    hist_cols = [c for c in daily.columns
                 if c.endswith('_lag1') or any(c.endswith(f'_r{w}') for w in windows)]
    df = df.merge(daily[['stock_id', 'date_id', *hist_cols]],
                  on=['stock_id', 'date_id'], how='left')
    return df

In [44]:

def build_all_features(df):
    df = build_features(df)          # your original: signed_imb, imb_ratio, book_imb, ...
    df = add_price_features(df)      # mid_price, near_wap, far_near, ref_depth, ...
    df = add_size_features(df)       # imb_to_matched, book_total, imb_flag_size, ...
    df = add_time_features(df)       # *_diff, *_ret, momentum, auction_frac  (re-sorts!)
    df = add_cross_sectional(df)     # *_rank, *_z across peers at each timestep
    df = add_historical_features(df)
    return df


df_feat = build_all_features(df)

/var/folders/84/plrgtnrj32j1fvbb6rkrw0r00000gn/T/ipykernel_82667/3197615508.py:7: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df[f'{col}_ret']    = g[col].pct_change()    # % change, scale-free
/var/folders/84/plrgtnrj32j1fvbb6rkrw0r00000gn/T/ipykernel_82667/3197615508.py:7: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df[f'{col}_ret']    = g[col].pct_change()    # % change, scale-free
/var/folders/84/plrgtnrj32j1fvbb6rkrw0r00000gn/T/ipykernel_82667/3197615508.py:7: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed i

In [54]:

train_fe, val_fe = train_val_split(df_feat, val_size=0.2)
y_train, y_val = train_fe['target'], val_fe['target']


NON_FEATURES = ['target', 'time_id', 'row_id', 'date_id']
FEATURES = [c for c in df_feat.columns if c not in NON_FEATURES]
print(f"{len(FEATURES)} features")
print(FEATURES)

Train set: 385 samples, Validation set: 96 samples
63 features
['stock_id', 'seconds_in_bucket', 'imbalance_size', 'imbalance_buy_sell_flag', 'reference_price', 'matched_size', 'far_price', 'near_price', 'bid_price', 'bid_size', 'ask_price', 'ask_size', 'wap', 'signed_imb', 'imb_ratio', 'book_imb', 'near_minus_wap', 'ref_minus_wap', 'spread', 'mid_price', 'ref_wap', 'near_wap', 'far_near', 'far_wap', 'ref_depth', 'wap_depth', 'imb_to_matched', 'book_total', 'auction_vs_book', 'imb_flag_size', 'wap_diff', 'wap_ret', 'imbalance_size_diff', 'imbalance_size_ret', 'reference_price_diff', 'reference_price_ret', 'imb_flag_size_diff', 'imb_flag_size_ret', 'matched_size_diff', 'matched_size_ret', 'wap_mom_60', 'imb_mom_60', 'auction_frac', 'imb_flag_size_rank', 'imb_flag_size_z', 'book_imb_rank', 'book_imb_z', 'near_wap_rank', 'near_wap_z', 'imb_to_matched_rank', 'imb_to_matched_z', 'day_target_std_lag1', 'day_target_std_r5', 'day_target_std_r20', 'day_target_absmean_lag1', 'day_target_absmean_

In [52]:
print(FEATURES)

['stock_id', 'date_id', 'seconds_in_bucket', 'imbalance_size', 'imbalance_buy_sell_flag', 'reference_price', 'matched_size', 'far_price', 'near_price', 'bid_price', 'bid_size', 'ask_price', 'ask_size', 'wap', 'target', 'time_id', 'row_id', 'signed_imb', 'imb_ratio', 'book_imb', 'near_minus_wap', 'ref_minus_wap', 'spread', 'mid_price', 'ref_wap', 'near_wap', 'far_near', 'far_wap', 'ref_depth', 'wap_depth', 'imb_to_matched', 'book_total', 'auction_vs_book', 'imb_flag_size', 'wap_diff', 'wap_ret', 'imbalance_size_diff', 'imbalance_size_ret', 'reference_price_diff', 'reference_price_ret', 'imb_flag_size_diff', 'imb_flag_size_ret', 'matched_size_diff', 'matched_size_ret', 'wap_mom_60', 'imb_mom_60', 'auction_frac', 'imb_flag_size_rank', 'imb_flag_size_z', 'book_imb_rank', 'book_imb_z', 'near_wap_rank', 'near_wap_z', 'imb_to_matched_rank', 'imb_to_matched_z', 'day_target_std_lag1', 'day_target_std_r5', 'day_target_std_r20', 'day_target_absmean_lag1', 'day_target_absmean_r5', 'day_target_absm

In [55]:
X_train = train_fe[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = val_fe[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)

model = lgb.LGBMRegressor(
    objective='mae', n_estimators=2000, learning_rate=0.03, num_leaves=128,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    min_child_samples=100, random_state=42, n_jobs=-1,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)], eval_metric='mae',
    categorical_feature=['stock_id'],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)
pred_lgb = model.predict(X_val)
print(f"LightGBM (all features) val MAE: {mae(y_val, pred_lgb):.4f}")

/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070519 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15358
[LightGBM] [Info] Number of data points in the train set: 4181948, number of used features: 63
[LightGBM] [Info] Start training from score -0.069737
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l1: 5.95227
[200]	valid_0's l1: 5.94155
[300]	valid_0's l1: 5.93893
[400]	valid_0's l1: 5.93798
[500]	valid_0's l1: 5.93759
Early stopping, best iteration is:
[496]	valid_0's l1: 5.93752
LightGBM (all features) val MAE: 5.9375


In [56]:
# ── Train vs Val gap check ───────────────────────────────────────────────
# You already scored on X_val. Now also score on the data the model trained on.
train_pred = model.predict(X_train)
val_pred   = model.predict(X_val)     # same as pred_lgb from before

train_mae = mae(y_train, train_pred)
val_mae   = mae(y_val,   val_pred)

print(f"train MAE: {train_mae:.4f}")
print(f"val   MAE: {val_mae:.4f}")
print(f"gap (val - train): {val_mae - train_mae:.4f}")

train MAE: 6.1884
val   MAE: 5.9375
gap (val - train): -0.2509


In [57]:
train_zero = mae(y_train, np.zeros(len(y_train)))
val_zero   = mae(y_val,   np.zeros(len(y_val)))

train_skill = (train_zero - train_mae) / train_zero * 100
val_skill   = (val_zero   - val_mae)   / val_zero   * 100

print(f"train: zero={train_zero:.4f}  model={train_mae:.4f}  skill={train_skill:.2f}%")
print(f"val:   zero={val_zero:.4f}  model={val_mae:.4f}  skill={val_skill:.2f}%")

train: zero=6.4956  model=6.1884  skill=4.73%
val:   zero=6.0601  model=5.9375  skill=2.02%


In [58]:
model_reg = lgb.LGBMRegressor(
    objective='mae', n_estimators=3000, learning_rate=0.03,
    num_leaves=31, min_child_samples=1000,
    reg_alpha=1.0, reg_lambda=1.0,
    subsample=0.7, subsample_freq=1, colsample_bytree=0.7,
    random_state=42, n_jobs=-1,
)
model_reg.fit(
    X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='mae',
    categorical_feature=['stock_id'],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

# Re-run the SAME skill diagnostic so it's an apples-to-apples comparison
tr_mae = mae(y_train, model_reg.predict(X_train))
va_mae = mae(y_val,   model_reg.predict(X_val))
print(f"train: model={tr_mae:.4f}  skill={(train_zero-tr_mae)/train_zero*100:.2f}%")
print(f"val:   model={va_mae:.4f}  skill={(val_zero-va_mae)/val_zero*100:.2f}%")

/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.065332 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15358
[LightGBM] [Info] Number of data points in the train set: 4181948, number of used features: 63
[LightGBM] [Info] Start training from score -0.069737
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 5.95106
[400]	valid_0's l1: 5.94385
[600]	valid_0's l1: 5.94128
[800]	valid_0's l1: 5.93988
[1000]	valid_0's l1: 5.93897
[1200]	valid_0's l1: 5.93851
[1400]	valid_0's l1: 5.93809
[1600]	valid_0's l1: 5.93769
Early stopping, best iteration is:
[1651]	valid_0's l1: 5.93762
train: model=6.2184  skill=4.27%
val:   model=5.9376  skill=2.02%


## K fold

In [59]:
# ── Walk-forward (expanding-window) validation ───────────────────────────
# Fold k trains on everything BEFORE its validation chunk, then validates on
# the next `val_days` of dates. The window expands; we never train on the
# future. Reporting SKILL (% over each fold's own zero-baseline) makes folds
# comparable even when some are calmer than others.
def walk_forward_validate(df_feat, features, params, n_folds=4, val_days=50):
    dates = np.sort(df_feat['date_id'].unique())
    results = []

    for k in range(n_folds):
        # Validation chunk for this fold, counting back from the end.
        # Fold 0 is the EARLIEST val chunk, so folds read left->right in time.
        val_end   = len(dates) - (n_folds - 1 - k) * val_days
        val_start = val_end - val_days
        val_dates   = dates[val_start:val_end]
        train_dates = dates[:val_start]                 # strictly before val

        tr = df_feat[df_feat['date_id'].isin(train_dates)]
        va = df_feat[df_feat['date_id'].isin(val_dates)]

        X_tr = tr[features].replace([np.inf, -np.inf], np.nan).fillna(0)
        X_va = va[features].replace([np.inf, -np.inf], np.nan).fillna(0)
        y_tr, y_va = tr['target'], va['target']

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='mae',
            categorical_feature=['stock_id'],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        # Per-fold skill = improvement over THIS fold's zero-baseline
        va_mae   = mae(y_va, model.predict(X_va))
        va_zero  = mae(y_va, np.zeros(len(y_va)))
        skill    = (va_zero - va_mae) / va_zero * 100

        results.append({
            'fold': k,
            'train_days': len(train_dates),
            'val_dates': f"{val_dates.min()}-{val_dates.max()}",
            'val_zero': va_zero, 'val_mae': va_mae, 'skill_%': skill,
            'best_iter': model.best_iteration_,
        })
        print(f"fold {k}: train={len(train_dates):>3}d  val {val_dates.min()}-{val_dates.max()}  "
              f"zero={va_zero:.4f}  mae={va_mae:.4f}  skill={skill:.2f}%")

    res = pd.DataFrame(results)
    print(f"\nMEAN skill: {res['skill_%'].mean():.2f}%  "
          f"(std {res['skill_%'].std():.2f})  |  MEAN val MAE: {res['val_mae'].mean():.4f}")
    return res

In [60]:
# ── Run it with your current (regularized) params on the full feature set ─
params = dict(
    objective='mae', n_estimators=3000, learning_rate=0.03,
    num_leaves=31, min_child_samples=1000,
    reg_alpha=1.0, reg_lambda=1.0,
    subsample=0.7, subsample_freq=1, colsample_bytree=0.7,
    random_state=42, n_jobs=-1, verbose=-1,
)

cv = walk_forward_validate(df_feat, FEATURES, params, n_folds=4, val_days=50)

/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 0: train=281d  val 281-330  zero=6.4861  mae=6.3600  skill=1.94%


/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 1: train=331d  val 331-380  zero=6.3966  mae=6.2614  skill=2.11%


/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 2: train=381d  val 381-430  zero=6.2211  mae=6.0873  skill=2.15%


/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 3: train=431d  val 431-480  zero=5.9341  mae=5.8134  skill=2.03%

MEAN skill: 2.06%  (std 0.09)  |  MEAN val MAE: 6.1305


In [61]:
# ── Zero-sum post-processing ─────────────────────────────────────────────
# The target is a stock's move RELATIVE to the index, so at each timestamp the
# cross-stock (weighted) mean is ~0 by construction. Our model doesn't know
# that. Subtracting the per-timestamp mean nudges predictions back onto that
# constraint. We test it INSIDE the CV so the number is trustworthy.
def apply_zero_sum(val_df, preds):
    # val_df must carry date_id + seconds_in_bucket aligned to preds (same order)
    tmp = val_df[['date_id', 'seconds_in_bucket']].copy()
    tmp['pred'] = preds
    # mean prediction across all stocks at this exact instant
    inst_mean = tmp.groupby(['date_id', 'seconds_in_bucket'])['pred'].transform('mean')
    return tmp['pred'].values - inst_mean.values


In [62]:
# ── Walk-forward (expanding-window) validation ───────────────────────────
# Fold k trains on everything BEFORE its validation chunk, then validates on
# the next `val_days` of dates. The window expands; we never train on the
# future. Reporting SKILL (% over each fold's own zero-baseline) makes folds
# comparable even when some are calmer than others.
def walk_forward_validate_adj(df_feat, features, params, n_folds=4, val_days=50):
    dates = np.sort(df_feat['date_id'].unique())
    results = []

    for k in range(n_folds):
        # Validation chunk for this fold, counting back from the end.
        # Fold 0 is the EARLIEST val chunk, so folds read left->right in time.
        val_end   = len(dates) - (n_folds - 1 - k) * val_days
        val_start = val_end - val_days
        val_dates   = dates[val_start:val_end]
        train_dates = dates[:val_start]                 # strictly before val

        tr = df_feat[df_feat['date_id'].isin(train_dates)]
        va = df_feat[df_feat['date_id'].isin(val_dates)]

        X_tr = tr[features].replace([np.inf, -np.inf], np.nan).fillna(0)
        X_va = va[features].replace([np.inf, -np.inf], np.nan).fillna(0)
        y_tr, y_va = tr['target'], va['target']

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='mae',
            categorical_feature=['stock_id'],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        raw   = model.predict(X_va)
        adj   = apply_zero_sum(va, raw)

        # Per-fold skill = improvement over THIS fold's zero-baseline
        va_mae   = mae(y_va, adj)
        va_zero  = mae(y_va, np.zeros(len(y_va)))
        skill    = (va_zero - va_mae) / va_zero * 100

        results.append({
            'fold': k,
            'train_days': len(train_dates),
            'val_dates': f"{val_dates.min()}-{val_dates.max()}",
            'val_zero': va_zero, 'val_mae': va_mae, 'skill_%': skill,
            'best_iter': model.best_iteration_,
        })
        print(f"fold {k}: train={len(train_dates):>3}d  val {val_dates.min()}-{val_dates.max()}  "
              f"zero={va_zero:.4f}  mae={va_mae:.4f}  skill={skill:.2f}%")

    res = pd.DataFrame(results)
    print(f"\nMEAN skill: {res['skill_%'].mean():.2f}%  "
          f"(std {res['skill_%'].std():.2f})  |  MEAN val MAE: {res['val_mae'].mean():.4f}")
    return res

In [63]:
params = dict(
    objective='mae', n_estimators=3000, learning_rate=0.03,
    num_leaves=31, min_child_samples=1000,
    reg_alpha=1.0, reg_lambda=1.0,
    subsample=0.7, subsample_freq=1, colsample_bytree=0.7,
    random_state=42, n_jobs=-1, verbose=-1,
)

cv = walk_forward_validate_adj(df_feat, FEATURES, params, n_folds=4, val_days=50)

/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 0: train=281d  val 281-330  zero=6.4861  mae=6.3580  skill=1.97%


/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 1: train=331d  val 331-380  zero=6.3966  mae=6.2613  skill=2.12%


/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 2: train=381d  val 381-430  zero=6.2211  mae=6.0867  skill=2.16%


/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 3: train=431d  val 431-480  zero=5.9341  mae=5.8091  skill=2.11%

MEAN skill: 2.09%  (std 0.08)  |  MEAN val MAE: 6.1288
